<a href="https://colab.research.google.com/github/Riyachhetrii/diabetes_analysis/blob/main/diabetes_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diabetes Dataset Analysis
### Phase 1 -- Data Collection & Preparation  
**Dataset:** Pima Indians Diabetes (Kaggle: `lara311/diabetes-dataset-using-many-medical-metrics`)


## Cell 1 -- Install dependencies

In [ ]:
# Run once; comment out after first run
import subprocess, sys
pkgs = ["pandas", "numpy", "matplotlib", "seaborn"]
subprocess.run([sys.executable, "-m", "pip", "install", *pkgs, "-q"], check=True)
print("Dependencies ready")

Dependencies ready


## Cell 2 -- Imports & global style

In [ ]:
import io
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="Set2", font_scale=1.05)
plt.rcParams.update({
    "figure.dpi"        : 120,
    "savefig.bbox"      : "tight",
    "savefig.dpi"       : 150,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
})

COLORS  = {"non_diabetic": "#4DBEEE", "diabetic": "#D95319"}
PALETTE = [COLORS["non_diabetic"], COLORS["diabetic"]]
FEATURES = [
    "Pregnancies", "Glucose", "BloodPressure",
    "SkinThickness", "Insulin", "BMI",
    "DiabetesPedigreeFunction", "Age",
]
print("Imports complete")

## Cell 3 -- Load data from public URL
The dataset is fetched automatically from a public GitHub mirror of the Pima Indians Diabetes dataset.
It is identical to the Kaggle version (`lara311/diabetes-dataset-using-many-medical-metrics`).
No API key or manual download is required.

In [ ]:
URL = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"

COLS = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome",
]

print(f"Fetching dataset from:\n  {URL}")
response = requests.get(URL, timeout=15)
response.raise_for_status()

df_raw = pd.read_csv(io.StringIO(response.text))
df_raw.columns = COLS[:len(df_raw.columns)]

print(f"Loaded {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
df_raw.head()

---
# PHASE 1 -- Data Preparation

## Cell 4 -- Raw dataset overview

In [ ]:
print(f"Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print(f"Columns: {list(df_raw.columns)}\n")
print(df_raw.dtypes.rename('dtype').to_string())
print()
df_raw.describe().round(2)

## Cell 5 -- Attribute descriptions

In [ ]:
attr_desc = {
    "Pregnancies"             : "Number of times pregnant (int, 0-17)",
    "Glucose"                 : "2-hr plasma glucose mg/dL (int)",
    "BloodPressure"           : "Diastolic blood pressure mmHg (int)",
    "SkinThickness"           : "Triceps skinfold thickness mm (int)",
    "Insulin"                 : "2-hr serum insulin mIU/L (int)",
    "BMI"                     : "Body mass index kg/m2 (float)",
    "DiabetesPedigreeFunction": "Genetic diabetes likelihood score (float)",
    "Age"                     : "Age in years (int, 21-81)",
    "Outcome"                 : "1 = diabetic, 0 = non-diabetic (binary)",
}
desc_df = pd.DataFrame.from_dict(attr_desc, orient="index", columns=["Description"])
desc_df.index.name = "Column"
desc_df

## Cell 6 -- Zero-value analysis (biologically impossible zeros)

In [ ]:
cols_with_invalid_zeros = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

rows = []
for col in cols_with_invalid_zeros:
    n   = (df_raw[col] == 0).sum()
    pct = n / len(df_raw) * 100
    rows.append({"Column": col, "Zero count": n, "Zero %": round(pct, 2),
                 "Decision": "Replace with column mean"})

zero_df = pd.DataFrame(rows).set_index("Column")

print("Biologically impossible zeros (Pregnancies & Outcome excluded):\n")
print(zero_df.to_string())
print("\nRationale:")
print("  A glucose reading of 0, blood pressure of 0, or BMI of 0 is")
print("  physically impossible -- these are missing values encoded as 0.")
print("  All such zeros are replaced with the non-zero column mean to")
print("  preserve the full 768-row sample size.")
print("  Pregnancies=0 is valid. Outcome=0/1 is the binary target -- no change.")

## Cell 7 -- Data cleaning: replace invalid zeros with column means

In [ ]:
df_clean = df_raw.copy()

print(f"{\"Column\":<28} {\"Zeros replaced\":>16}  {\"Imputed mean\":>14}")
print("-" * 62)

for col in cols_with_invalid_zeros:
    col_mean   = df_clean.loc[df_clean[col] != 0, col].mean()
    n_replaced = (df_clean[col] == 0).sum()
    df_clean[col] = df_clean[col].replace(0, col_mean)
    print(f"  {col:<26} {n_replaced:>16}  {col_mean:>14.4f}")

print("\nCleaning complete")

## Cell 8 -- Post-cleaning validation

In [ ]:
print("Remaining zeros in cleaned columns:")
for col in cols_with_invalid_zeros:
    count  = (df_clean[col] == 0).sum()
    status = "OK" if count == 0 else "STILL HAS ZEROS"
    print(f"  {col:<22}: {count}  {status}")

nan_count = df_clean.isnull().sum().sum()
print(f"\nTotal NaN values: {nan_count}  {'OK' if nan_count == 0 else 'FOUND NaNs'}")

n_diabetic     = (df_clean["Outcome"] == 1).sum()
n_non_diabetic = (df_clean["Outcome"] == 0).sum()
n_total        = len(df_clean)
print(f"\nClass balance:")
print(f"  Non-diabetic (0): {n_non_diabetic}  ({n_non_diabetic/n_total*100:.1f}%)")
print(f"  Diabetic     (1): {n_diabetic}  ({n_diabetic/n_total*100:.1f}%)")
print(f"\nFinal dataset size: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns")

## Cell 9 -- Summary statistics (cleaned dataset)

In [ ]:
df_clean.describe().round(2)

## Cell 10 -- Export cleaned CSV

In [ ]:
OUTPUT_PATH = "diabetes_clean.csv"
df_clean.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned dataset saved to: {OUTPUT_PATH}")
print(f"  {df_clean.shape[0]} rows x {df_clean.shape[1]} columns")
print("\nPhase 1 complete.")

---
# PHASE 2 -- Exploratory Data Analysis
The cleaned `df_clean` dataframe from Phase 1 flows directly into all cells below.
Eight figures are generated and saved as PNGs ready to embed in the IEEE report.

## Cell 11 -- Split by outcome class

In [ ]:
diabetic     = df_clean[df_clean["Outcome"] == 1].copy()
non_diabetic = df_clean[df_clean["Outcome"] == 0].copy()
print(f"Non-diabetic: {len(non_diabetic)}  |  Diabetic: {len(diabetic)}  |  Total: {len(df_clean)}")

## Cell 12 -- Summary statistics by class

In [ ]:
stats = pd.concat([
    df_clean.groupby("Outcome")[FEATURES].mean().T
        .rename(columns={0: "Mean (non-diabetic)", 1: "Mean (diabetic)"}),
    df_clean.groupby("Outcome")[FEATURES].std().T
        .rename(columns={0: "Std (non-diabetic)", 1: "Std (diabetic)"}),
], axis=1).round(2)
stats

## Cell 13 -- Figure 1: Feature distributions
Overlapping histograms for all 8 features split by class. Dashed lines mark group means.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle("Figure 1 -- Feature Distributions by Diabetes Outcome", fontsize=14, y=1.01)

for ax, feat in zip(axes.flatten(), FEATURES):
    ax.hist(non_diabetic[feat], bins=25, alpha=0.65, color=COLORS["non_diabetic"],
            label=f"Non-diabetic (mean={non_diabetic[feat].mean():.1f})",
            density=True, edgecolor="none")
    ax.hist(diabetic[feat], bins=25, alpha=0.65, color=COLORS["diabetic"],
            label=f"Diabetic (mean={diabetic[feat].mean():.1f})",
            density=True, edgecolor="none")
    ax.axvline(non_diabetic[feat].mean(), color=COLORS["non_diabetic"],
               linestyle="--", linewidth=1.4)
    ax.axvline(diabetic[feat].mean(), color=COLORS["diabetic"],
               linestyle="--", linewidth=1.4)
    ax.set_title(feat, fontsize=11)
    ax.set_xlabel("Value", fontsize=9)
    ax.set_ylabel("Density", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=7, frameon=False)

plt.tight_layout()
plt.savefig("fig1_feature_distributions.png")
plt.show()
print("Saved: fig1_feature_distributions.png")

## Cell 14 -- Figure 2: Box plots by class
Side-by-side box plots for every feature. Supports visual inspection of H1 (Glucose), H2 (BMI, Age), and H3 (Pregnancies).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle("Figure 2 -- Feature Box Plots: Diabetic vs Non-Diabetic", fontsize=14, y=1.01)

for ax, feat in zip(axes.flatten(), FEATURES):
    bp = ax.boxplot(
        [non_diabetic[feat].values, diabetic[feat].values],
        patch_artist=True, widths=0.5,
        medianprops=dict(color="white", linewidth=2),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2),
        flierprops=dict(marker="o", markersize=3, alpha=0.4),
    )
    for patch, color in zip(bp["boxes"], PALETTE):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["Non-diabetic", "Diabetic"], fontsize=9)
    ax.set_title(feat, fontsize=11)
    ax.set_ylabel("Value", fontsize=9)
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig("fig2_boxplots.png")
plt.show()
print("Saved: fig2_boxplots.png")

## Cell 15 -- Figure 3: Correlation heatmap
Lower-triangle heatmap. The **Outcome** row shows which features correlate most strongly with diabetes -- directly justifies the hypothesis selection.

In [ ]:
corr = df_clean[FEATURES + ["Outcome"]].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(220, 10, as_cmap=True)

sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", linewidths=0.5,
    cmap=cmap, center=0, vmin=-1, vmax=1,
    annot_kws={"size": 10}, ax=ax,
)
ax.set_title("Figure 3 -- Correlation Heatmap (all features + Outcome)", fontsize=13, pad=14)
ax.tick_params(axis="x", rotation=35, labelsize=10)
ax.tick_params(axis="y", rotation=0,  labelsize=10)

plt.tight_layout()
plt.savefig("fig3_correlation_heatmap.png")
plt.show()
print("Saved: fig3_correlation_heatmap.png")

print("\nFeature correlations with Outcome (descending):")
outcome_corr = corr["Outcome"].drop("Outcome").sort_values(ascending=False)
for feat, val in outcome_corr.items():
    bar = chr(9608) * int(abs(val) * 30)
    print(f"  {feat:<28}: {val:+.4f}  {bar}")

## Cell 16 -- Figure 4: Pairplot (top 4 predictors)
Glucose, BMI, Age, and DiabetesPedigreeFunction plotted pairwise. `corner=True` removes redundant upper triangle.

In [ ]:
top4 = ["Glucose", "BMI", "Age", "DiabetesPedigreeFunction", "Outcome"]
pair_df = df_clean[top4].copy()
pair_df["Outcome"] = pair_df["Outcome"].map({0: "Non-diabetic", 1: "Diabetic"})

g = sns.pairplot(
    pair_df, hue="Outcome",
    palette={"Non-diabetic": COLORS["non_diabetic"], "Diabetic": COLORS["diabetic"]},
    plot_kws={"alpha": 0.4, "s": 18},
    diag_kind="kde",
    corner=True,
)
g.figure.suptitle("Figure 4 -- Pairplot: Top 4 Predictors", y=1.01, fontsize=13)
g.figure.savefig("fig4_pairplot.png")
plt.show()
print("Saved: fig4_pairplot.png")

## Cell 17 -- Figure 5: Outcome class distribution

In [ ]:
counts = df_clean["Outcome"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle("Figure 5 -- Outcome Class Distribution", fontsize=13)

axes[0].bar(["Non-diabetic (0)", "Diabetic (1)"], counts.values,
            color=PALETTE, alpha=0.85, width=0.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 4, str(v), ha="center", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].set_ylim(0, counts.max() * 1.15)
axes[0].set_title("Count by class")

labels = [f"Non-diabetic\n({counts[0]/len(df_clean)*100:.1f}%)",
          f"Diabetic\n({counts[1]/len(df_clean)*100:.1f}%)"]
axes[1].pie(counts.values, labels=labels, colors=PALETTE,
            startangle=90, wedgeprops=dict(edgecolor="white", linewidth=2),
            textprops={"fontsize": 11})
axes[1].set_title("Proportion")

plt.tight_layout()
plt.savefig("fig5_outcome_distribution.png")
plt.show()
print("Saved: fig5_outcome_distribution.png")

## Cell 18 -- Figure 6: Glucose by outcome (supports H1)
**H1:** Patients with diabetes have significantly higher glucose levels than non-diabetic patients.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(non_diabetic["Glucose"], bins=30, color=COLORS["non_diabetic"], alpha=0.7,
        label=f"Non-diabetic  (mean = {non_diabetic['Glucose'].mean():.1f} mg/dL)",
        density=True, edgecolor="none")
ax.hist(diabetic["Glucose"], bins=30, color=COLORS["diabetic"], alpha=0.7,
        label=f"Diabetic       (mean = {diabetic['Glucose'].mean():.1f} mg/dL)",
        density=True, edgecolor="none")
ax.axvline(non_diabetic["Glucose"].mean(), color=COLORS["non_diabetic"],
           linestyle="--", linewidth=2.0)
ax.axvline(diabetic["Glucose"].mean(), color=COLORS["diabetic"],
           linestyle="--", linewidth=2.0)

ax.set_title("Figure 6 -- Glucose Levels by Diabetes Outcome  (supports H1)", fontsize=13)
ax.set_xlabel("Plasma Glucose (mg/dL)")
ax.set_ylabel("Density")
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig("fig6_glucose_by_outcome.png")
plt.show()

diff     = diabetic["Glucose"].mean() - non_diabetic["Glucose"].mean()
corr_g   = df_clean["Glucose"].corr(df_clean["Outcome"])
print(f"Mean difference : +{diff:.2f} mg/dL")
print(f"Correlation r   : {corr_g:.4f}")
print("Saved: fig6_glucose_by_outcome.png")

## Cell 19 -- Figure 7: BMI vs Age scatter (supports H2)
**H2:** Higher BMI combined with older age is associated with a greater likelihood of diabetes. Lines are linear trend fits per class.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for outcome, label, color in [
    (0, "Non-diabetic", COLORS["non_diabetic"]),
    (1, "Diabetic",     COLORS["diabetic"]),
]:
    sub = df_clean[df_clean["Outcome"] == outcome]
    ax.scatter(sub["Age"], sub["BMI"], alpha=0.40, s=22, color=color, label=label)
    z = np.polyfit(sub["Age"], sub["BMI"], 1)
    x_range = np.linspace(df_clean["Age"].min(), df_clean["Age"].max(), 100)
    ax.plot(x_range, np.poly1d(z)(x_range), color=color, linewidth=2.2)

ax.set_title("Figure 7 -- BMI vs Age by Diabetes Outcome  (supports H2)", fontsize=13)
ax.set_xlabel("Age (years)")
ax.set_ylabel("BMI (kg/m2)")
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig("fig7_bmi_age_scatter.png")
plt.show()

bmi_diff = diabetic["BMI"].mean() - non_diabetic["BMI"].mean()
age_diff = diabetic["Age"].mean() - non_diabetic["Age"].mean()
print(f"Mean BMI -- non-diabetic: {non_diabetic['BMI'].mean():.2f}  |  diabetic: {diabetic['BMI'].mean():.2f}  (delta {bmi_diff:+.2f})")
print(f"Mean Age -- non-diabetic: {non_diabetic['Age'].mean():.2f}  |  diabetic: {diabetic['Age'].mean():.2f}  (delta {age_diff:+.2f})")
print("Saved: fig7_bmi_age_scatter.png")

## Cell 20 -- Figure 8: Diabetes rate by pregnancies (supports H3)
**H3:** Patients with more pregnancies show a higher rate of diabetes diagnosis. Bar height = percentage of diabetic patients in that pregnancy group.

In [ ]:
df_clean["Preg_group"] = pd.cut(
    df_clean["Pregnancies"],
    bins=[-1, 0, 2, 5, 10, 20],
    labels=["0", "1-2", "3-5", "6-10", "11+"],
)

preg_rate = (
    df_clean.groupby("Preg_group", observed=True)["Outcome"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "diabetes_rate", "count": "n"})
    .reset_index()
)
preg_rate["diabetes_rate"] *= 100

overall_rate = df_clean["Outcome"].mean() * 100

fig, ax = plt.subplots(figsize=(9, 5))
bar_colors = [
    COLORS["diabetic"] if r >= overall_rate else COLORS["non_diabetic"]
    for r in preg_rate["diabetes_rate"]
]
bars = ax.bar(
    preg_rate["Preg_group"].astype(str),
    preg_rate["diabetes_rate"],
    color=bar_colors, alpha=0.85, width=0.55,
)
for bar, (_, row) in zip(bars, preg_rate.iterrows()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.8,
        f"{row['diabetes_rate']:.1f}%\n(n={int(row['n'])})",
        ha="center", va="bottom", fontsize=9,
    )

ax.axhline(overall_rate, color="gray", linestyle="--", linewidth=1.4,
           label=f"Overall rate ({overall_rate:.1f}%)")
ax.set_title("Figure 8 -- Diabetes Rate by Number of Pregnancies  (supports H3)", fontsize=13)
ax.set_xlabel("Number of Pregnancies")
ax.set_ylabel("Diabetes Rate (%)")
ax.set_ylim(0, preg_rate["diabetes_rate"].max() * 1.25)
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig("fig8_pregnancies_diabetes_rate.png")
plt.show()
print(preg_rate.to_string(index=False))
print("Saved: fig8_pregnancies_diabetes_rate.png")

## Cell 21 -- Hypothesis support summary

In [ ]:
corr_matrix  = df_clean[FEATURES + ["Outcome"]].corr()["Outcome"].drop("Outcome")

glucose_diff = diabetic["Glucose"].mean() - non_diabetic["Glucose"].mean()
bmi_diff     = diabetic["BMI"].mean()     - non_diabetic["BMI"].mean()
age_diff     = diabetic["Age"].mean()     - non_diabetic["Age"].mean()
preg_diff    = diabetic["Pregnancies"].mean() - non_diabetic["Pregnancies"].mean()

summary = pd.DataFrame([
    {
        "Hypothesis" : "H1 -- Glucose",
        "Key finding": f"Diabetic mean {diabetic['Glucose'].mean():.1f} vs {non_diabetic['Glucose'].mean():.1f} mg/dL (delta +{glucose_diff:.1f})",
        "Correlation": f"r = {corr_matrix['Glucose']:+.4f}",
        "Verdict"    : "SUPPORTED" if glucose_diff > 15 else "Partially supported",
    },
    {
        "Hypothesis" : "H2 -- BMI + Age",
        "Key finding": f"BMI delta +{bmi_diff:.2f}  |  Age delta +{age_diff:.2f} yrs",
        "Correlation": f"BMI r={corr_matrix['BMI']:+.4f}  Age r={corr_matrix['Age']:+.4f}",
        "Verdict"    : "SUPPORTED" if bmi_diff > 1 and age_diff > 1 else "Partially supported",
    },
    {
        "Hypothesis" : "H3 -- Pregnancies",
        "Key finding": f"Mean pregnancies delta +{preg_diff:.2f}",
        "Correlation": f"r = {corr_matrix['Pregnancies']:+.4f}",
        "Verdict"    : "MODERATELY SUPPORTED",
    },
])
summary.set_index("Hypothesis")

## Cell 22 -- Output files

In [ ]:
import glob, os

print("Files generated:\n")
for pattern in ["diabetes_clean.csv", "fig*.png"]:
    for f in sorted(glob.glob(pattern)):
        size_kb = os.path.getsize(f) / 1024
        print(f"  {f:<45} {size_kb:>7.1f} KB")

print("\nPhase 1 + Phase 2 complete.")
print("  diabetes_clean.csv -- submit as your data file")
print("  fig1-8 PNGs        -- embed into your IEEE report")